# Actividad 27/04/2026

## Bootstrap

1- Tomar dataset Motor Trend Road 

2- Realizar regresion linel sobre m^pg = B_0^ + B_1^(hp)+b_2^(qsec) de los datos 
-2a Calcular intervalos de confianza de los betas  

3- Crear 1000 muestras de bootstrap con m  observaciones con reemplazo 
-3a realizar la regresion  m^pg= B_0^ + B_1^(hp)+b_2^(qsec)
-3b Calcular B_x y O_B abrir intervalos de intervalos 

4- Comparar resultados entre 2 y 3 


In [131]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from scipy.stats import t
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.ensemble import GradientBoostingRegressor




### 1- Tomar dataset Motor Trend Road 

In [132]:


obj = pd.read_excel("Motor Trend Car Road Tests.xlsx")
obj.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2



### 2- Realizar regresion linel sobre m^pg = B_0^ + B_1^(hp)+b_2^(qsec) de los datos 


In [133]:
X = obj[["hp", "qsec"]]
y = obj["mpg"]

lr = LinearRegression()
lr.fit(X,y)
B0 = lr.intercept_
B1 = lr.coef_[0]
B2 = lr.coef_[1]



print("B0:", B0)
print("B1 hp:", B1)
print("B2 qsec:", B2)



B0: 48.32370516913444
B1 hp: -0.0845930436740927
B2 qsec: -0.886579624634272


### -2a Calcular intervalos de confianza de los betas  


In [134]:
X_ols = sm.add_constant(X)

modelo_ols = sm.OLS(y, X_ols).fit()
print(modelo_ols.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     25.43
Date:              lun., 04 may. 2026   Prob (F-statistic):           4.18e-07
Time:                        17:28:58   Log-Likelihood:                -86.170
No. Observations:                  32   AIC:                             178.3
Df Residuals:                      29   BIC:                             182.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         48.3237     11.103      4.352      0.0

### 3- Crear 1000 muestras de bootstrap con m  observaciones con reemplazo 
### -3a realizar la regresion  m^pg= B_0^ + B_1^(hp)+b_2^(qsec)


In [135]:
B = 1000
m = len(obj)

betas_bootstrap = []

for i in range(B):
    muestra = obj.sample(n=m, replace=True)
    
    X_boot = muestra[["hp", "qsec"]]
    y_boot = muestra["mpg"]
    
    lr_boot = LinearRegression()
    lr_boot.fit(X_boot, y_boot)
    
    beta_boot = [
        lr_boot.intercept_,
        lr_boot.coef_[0],
        lr_boot.coef_[1]
    ]
    
    betas_bootstrap.append(beta_boot)

betas_bootstrap = np.array(betas_bootstrap)

### -3b Calcular B_x y O_B abrir intervalos de intervalos 


In [136]:
B_x = betas_bootstrap.mean(axis=0)
O_B = betas_bootstrap.std(axis=0)

lim_inf_boot = np.percentile(betas_bootstrap, 2.5, axis=0)
lim_sup_boot = np.percentile(betas_bootstrap, 97.5, axis=0)

intervalos_bootstrap = pd.DataFrame({
    "Coeficiente": ["B0", "B1_hp", "B2_qsec"],
    "B_x_bootstrap": B_x,
    "O_B_bootstrap": O_B,
    "Lim_inf_bootstrap": lim_inf_boot,
    "Lim_sup_bootstrap": lim_sup_boot
})

intervalos_bootstrap

,Coeficiente,B_x_bootstrap,O_B_bootstrap,Lim_inf_bootstrap,Lim_sup_bootstrap
0,B0,50.109684,10.805888,31.859145,74.265884
1,B1_hp,-0.087898,0.016531,-0.122692,-0.056895
2,B2_qsec,-0.965181,0.516924,-2.113878,-0.105687


###  4- Comparar resultados entre 2 y 3 


In [137]:
comparacion = pd.DataFrame({
    "Coeficiente": ["B0", "B1_hp", "B2_qsec"],
    
    "Bootstrap": B_x,
    
    "OLS": modelo_ols.params.values,
    
    "Bootstrap_Lim_inf": lim_inf_boot,
    "Bootstrap_Lim_sup": lim_sup_boot,
    
    "OLS_Lim_inf": modelo_ols.conf_int().iloc[:, 0].values,
    "OLS_Lim_sup": modelo_ols.conf_int().iloc[:, 1].values
})

comparacion

,Coeficiente,Bootstrap,OLS,Bootstrap_Lim_inf,Bootstrap_Lim_sup,OLS_Lim_inf,OLS_Lim_sup
0,B0,50.109684,48.323705,31.859145,74.265884,25.614894,71.032516
1,B1_hp,-0.087898,-0.084593,-0.122692,-0.056895,-0.113089,-0.056097
2,B2_qsec,-0.965181,-0.886580,-2.113878,-0.105687,-1.979929,0.206770


## Aggregating 


1. Cargar dataset Motor Trend.
2. Split: 50% train / 50% test (y = mpg).
3. Repetir 1000 veces:
   - Elegir 3 variables al azar
   - Entrenar modelo con train
   - Predecir en test
   - Calcular R^2 test
4. Guardar resultados.
5. Comparar y elegir mejor modelo.

In [138]:
X = obj.drop(columns=["model", "mpg"])
y = obj["mpg"]

X.head()

,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,8,360.0,175,3.15,3.440,17.02,0,0,3,2


###  Dividir los datos 50% train y 50% test


In [139]:


X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.50,random_state=42)

### 1000 modelos con 3 variables al azar 

In [140]:

np.random.seed(42)

B = 1000

resultados = []
predicciones = []

for i in range(B):
    
    varsi = np.random.choice(X_train.columns, size=3, replace=False)
    
    Xi = X_train[varsi]
    
    modelo_i = LinearRegression()
    modelo_i.fit(Xi, y_train)
    
    Xtesti = X_test[varsi]
    
    y_hat_i = modelo_i.predict(Xtesti)
    
    predicciones.append(y_hat_i)
    
    R2_test_i = r2_score(y_test, y_hat_i)
    
    resultados.append([
        i + 1,
        list(varsi),
        R2_test_i
    ])

### Resultados

In [141]:

resultados = pd.DataFrame(
    resultados,
    columns=["Modelo", "Variables", "R2_test"]
)

resultados

,Modelo,Variables,R2_test
0,1,"[gear, disp, qsec]",0.537990
1,2,"[cyl, disp, gear]",0.610802
2,3,"[carb, hp, cyl]",0.691024
3,4,"[disp, am, vs]",0.628576
4,5,"[disp, qsec, wt]",0.598845
...,...,...,...
995,996,"[cyl, hp, drat]",0.697297
996,997,"[drat, carb, vs]",0.442008
997,998,"[vs, carb, cyl]",0.714597
998,999,"[vs, gear, carb]",0.637766


### Mayor r^2

In [142]:

resultados_ordenados = resultados.sort_values(
    by="R2_test",
    ascending=False
)

resultados_ordenados.head(10)

,Modelo,Variables,R2_test
753,754,"[carb, disp, am]",0.777496
956,957,"[carb, am, disp]",0.777496
988,989,"[carb, am, disp]",0.777496
21,22,"[am, carb, disp]",0.777496
69,70,"[disp, carb, am]",0.777496
767,768,"[disp, carb, am]",0.777496
268,269,"[carb, cyl, am]",0.774560
724,725,"[carb, cyl, am]",0.774560
298,299,"[carb, cyl, am]",0.774560
841,842,"[am, carb, cyl]",0.774560


### Mejor modelo

In [143]:
mejor_modelo = resultados_ordenados.iloc[0]


print("Mejor modelo:", mejor_modelo["Modelo"])
print("Variables usadas:", mejor_modelo["Variables"])
print("R^2 test:", mejor_modelo["R2_test"])

Mejor modelo: 754
Variables usadas: ['carb', 'disp', 'am']
R^2 test: 0.777496034744581


### Promedio de predicciones de los 1000 modelos

In [144]:
predicciones = np.array(predicciones)

y_hat_prom = np.mean(predicciones, axis=0)

y_hat_prom

array([20.48228523, 10.23305087, 14.39487848, 27.14308243, 23.56425522,
       20.21908888, 13.62253977, 27.4748451 , 15.29199785, 21.8800055 ,
       15.39790621, 10.42073085, 19.78731969, 15.25967731, 14.76489001,
       13.36641253])

# Random Forest

In [145]:


# modelo
model = RandomForestRegressor(random_state=42)

# kfold
kf = KFold(n_splits=10, shuffle=True, random_state=42)


# parametros para gridsearch combinaciones 
param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [5, 10, None],
    'min_samples_leaf': [1, 2]
}

# gridsearch
grid = GridSearchCV(
    model,
    param_grid,
    cv=kf,
    scoring='r2',
    n_jobs=-1
)

# entrenar
grid.fit(X_train, y_train)

print("Mejores parámetros:", grid.best_params_)
print("Mejor R^2 (kfold):", grid.best_score_)

# mejor modelo
best_model = grid.best_estimator_

# fit final
best_model.fit(X_train, y_train)

# predicción
y_pred = best_model.predict(X_test)

# r2 final
r2 = r2_score(y_test, y_pred)

print("R^2 final:", r2)

C:\Users\sebas\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_search.py:1135: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


Mejores parámetros: {'max_depth': 5, 'min_samples_leaf': 1, 'n_estimators': 100}
Mejor R^2 (kfold): nan
R^2 final: 0.8372059628911183


# Boosting 

In [147]:

model_boosting = GradientBoostingRegressor(random_state=42)

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# grid paramestros
param_grid = {
    'n_estimators': [100, 300],
    'learning_rate': [0.01, 0.1],
    'max_depth': [2, 3, 5],
    'min_samples_leaf': [1, 2]
}

# gridsearch
grid_boosting = GridSearchCV(
    model_boosting,
    param_grid,
    cv=kf,
    scoring='r2',
    n_jobs=-1
)

# entrenar con train
grid_boosting.fit(X_train, y_train)

print("Mejores parámetros:")
print(grid_boosting.best_params_)

print("Mejor R^2 con KFold:")
print(grid_boosting.best_score_)

# mejor modelo
best_boosting = grid_boosting.best_estimator_

best_boosting.fit(X_train, y_train)

# predicción
y_pred_boosting = best_boosting.predict(X_test)

r2_boosting = r2_score(y_test, y_pred_boosting)

print("R^2 final en test:")
print(r2_boosting)

Mejores parámetros:
{'learning_rate': 0.01, 'max_depth': 2, 'min_samples_leaf': 1, 'n_estimators': 100}
Mejor R^2 con KFold:
nan
R^2 final en test:
0.7518290142733637


C:\Users\sebas\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\model_selection\_search.py:1135: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan]
  warnings.warn(
